# Module 2 · Notebook 2 — Eigen, SVD & PCA
### Computing, Robotics & Bionics · companion to A. Géron, *Hands-On Machine Learning with Scikit-Learn and PyTorch*

A **self-contained** continuation of Notebook 1: the two factorisations that power machine learning — the
**eigendecomposition** and the **singular value decomposition** — and the two payoffs they unlock,
**least squares** (Géron Ch. 4) and **PCA** (Ch. 7). Least squares is developed as what it geometrically
is, an orthogonal **projection** onto the column space, with the pseudoinverse that every library
actually calls and ridge regularisation read through the eigenvalues. Worked on real biomedical data
and a real image.

Run in **Google Colab** (*Runtime → Run all*). Everything is offline (the image and datasets ship with
Scikit-Learn).

**Contents**
1. Eigenvalues and eigenvectors (invariant directions)
2. Diagonalisation
3. Symmetric matrices and the spectral theorem
4. The SVD: rotate–scale–rotate
5. Low-rank image compression (bridge to Module 4)
6. Least squares as a projection, the pseudoinverse, and ridge (Ch. 4)
7. PCA on the breast-cancer data (Ch. 7)

Every section ends with an **Exercise**; run the **Solution** cell to check.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
np.set_printoptions(precision=3, suppress=True)
print("ready")

---
## 1 · Eigenvalues and eigenvectors

An **eigenvector** is a direction the matrix only *stretches*, not turns: `A v = lambda v`. We compute
them with `np.linalg.eig`, then *see* them: a circle of vectors maps to an ellipse, and the eigenvectors
are the directions that stay on their own line.

In [ ]:
A = np.array([[2.0, 1.0],
              [1.0, 2.0]])
vals, vecs = np.linalg.eig(A)
print("eigenvalues :", vals)            # 3 and 1
print("eigenvectors (columns):\n", vecs)
# check A v = lambda v for the first eigenpair
print("A v0 =", A @ vecs[:, 0], " vs  lambda0 v0 =", vals[0] * vecs[:, 0])

In [ ]:
# Visualise: unit circle -> ellipse; eigenvectors stay on their line
t = np.linspace(0, 2*np.pi, 200)
circle = np.array([np.cos(t), np.sin(t)])
ellipse = A @ circle
fig, ax = plt.subplots(figsize=(5,5))
ax.plot(*circle, color="lightgray", label="unit circle")
ax.plot(*ellipse, color="steelblue", label="A(circle)")
for i in range(2):
    v = vecs[:, i]
    ax.quiver(0,0,*v, angles="xy",scale_units="xy",scale=1,color="crimson",width=0.012)
    ax.quiver(0,0,*(A@v), angles="xy",scale_units="xy",scale=1,color="darkgreen",width=0.008)
ax.set_aspect("equal"); ax.grid(alpha=0.3); ax.legend()
ax.set_title("Eigenvectors (red) map to themselves scaled (green)"); plt.show()

**Exercise 1.** A simple two-compartment drug model transfers concentration between blood and tissue each time step via `A = np.array([[0.9, 0.1], [0.2, 0.8]])`. Find its eigenvalues and eigenvectors, and identify which eigenvalue corresponds to the slower-decaying mode (the one nearer 1) — that is the long-term direction the concentrations settle into.

> 🤖 *Gemini tip:* "Given a 2x2 NumPy matrix that models transfer between two compartments, show me how to find its eigenvalues/eigenvectors with np.linalg.eig and explain what the eigenvalue closest to 1 means physically."

In [ ]:
Acomp = np.array([[0.9, 0.1], [0.2, 0.8]])

# Your code here

---
## 2 · Diagonalisation

If `A` has independent eigenvectors, `A = P @ diag(lambda) @ inv(P)`: in the eigenvector basis the map is
pure scaling. This makes matrix powers trivial.

In [ ]:
P = vecs
Lam = np.diag(vals)
A_rebuilt = P @ Lam @ np.linalg.inv(P)
print("reconstructed A:\n", A_rebuilt, "\nmatches:", np.allclose(A, A_rebuilt))
# powers via diagonalisation: A^5
A5 = P @ np.diag(vals**5) @ np.linalg.inv(P)
print("A^5 matches np.linalg.matrix_power:", np.allclose(A5, np.linalg.matrix_power(A, 5)))

**Exercise 2.** Using the same compartment matrix `Acomp` from Exercise 1, use diagonalisation to predict the concentration state after 20 time steps starting from an initial state `x0 = np.array([1.0, 0.0])` (all drug starts in blood): compute `A^20 @ x0` via `P @ diag(lambda^20) @ inv(P) @ x0`, and confirm it matches `np.linalg.matrix_power(A, 20) @ x0`.

> 🤖 *Gemini tip:* "Show me how to use an eigendecomposition to compute A to a high power applied to a starting vector, and confirm it matches np.linalg.matrix_power."

In [ ]:
Acomp = np.array([[0.9, 0.1], [0.2, 0.8]])   # same compartment matrix as Exercise 1
x0 = np.array([1.0, 0.0])

# Your code here

---
## 3 · Symmetric matrices and the spectral theorem

If `A` is symmetric, its eigenvalues are real and its eigenvectors are **orthonormal** — use `eigh` (faster
and more accurate than `eig` for this case). Covariance matrices are symmetric, which is why PCA's axes
come out orthogonal.

In [ ]:
S = np.array([[2.0, 1.0], [1.0, 2.0]])     # symmetric
w, Q = np.linalg.eigh(S)
print("eigenvalues:", w)
print("eigenvectors orthonormal? Q.T@Q = I:", np.allclose(Q.T @ Q, np.eye(2)))
print("dot of the two eigenvectors (should be 0):", round(Q[:,0] @ Q[:,1], 6))

**Exercise 3.** Two EMG channels on neighbouring muscles share some crosstalk, giving the symmetric
covariance matrix `S = [[4,1],[1,4]]` — variance 4 in each channel, covariance 1 between them. Find
its eigenvalues with `eigh`, confirm the eigenvectors are orthogonal, and reconstruct `S` as
`Q @ diag(w) @ Q.T`. In one line, say what the two eigenvectors mean for the pair of channels.

> 🤖 *Gemini tip:* "Given a symmetric covariance matrix in NumPy, show me how to get its eigenvalues and orthonormal eigenvectors with eigh, how to check the reconstruction Q @ diag(w) @ Q.T, and what the eigenvectors of a covariance matrix represent."


In [ ]:
S2 = np.array([[4.0,1.0],[1.0,4.0]])

# Your code here

---
## 4 · The SVD: rotate–scale–rotate

The **singular value decomposition** `A = U @ diag(S) @ Vt` works for *any* matrix. Geometrically it is a
rotation, an axis-aligned scaling by the **singular values**, then another rotation — so the unit circle
maps to an ellipse whose semi-axis lengths are the singular values.

In [ ]:
M = np.array([[2.0, 1.0], [0.0, 1.5]])
U, S, Vt = np.linalg.svd(M)
print("singular values:", S)
print("U orthogonal:", np.allclose(U.T@U, np.eye(2)), "| V orthogonal:", np.allclose(Vt@Vt.T, np.eye(2)))
ell = M @ circle
fig, ax = plt.subplots(figsize=(5,5))
ax.plot(*circle, color="lightgray", label="unit circle")
ax.plot(*ell, color="steelblue", label="M(circle)")
for i in range(2):                              # semi-axes = sigma_i * u_i
    ax.quiver(0,0,*(S[i]*U[:,i]), angles="xy",scale_units="xy",scale=1,color="crimson",width=0.012)
ax.set_aspect("equal"); ax.grid(alpha=0.3); ax.legend()
ax.set_title("Singular values = ellipse semi-axis lengths"); plt.show()

**Exercise 4.** A 3-electrode EMG headset mixes two underlying muscle-activity sources through `M = np.array([[0.9, 0.2], [0.1, 0.8], [0.5, 0.5]])` (a 3x2 "mixing" matrix, one row per electrode). Compute its SVD and report the singular values — the larger one tells you the dominant, most energetic mixing direction.

> 🤖 *Gemini tip:* "Given a non-square NumPy matrix that mixes two signal sources into several sensor channels, show me how to compute its SVD and interpret the singular values."

In [ ]:
Memg = np.array([[0.9, 0.2], [0.1, 0.8], [0.5, 0.5]])

# Your code here

---
## 5 · Low-rank image compression (bridge to Module 4)

Keeping only the largest `k` singular values gives the **best** rank-`k` approximation of a matrix
(Eckart–Young). An image is just a matrix of pixel intensities, so SVD compresses it: most of the picture
lives in a few singular values. (Here a sample photo that ships with Scikit-Learn; in Module 4 you will do
the same on **Broad BBBC microscopy** images.)

In [ ]:
from sklearn.datasets import load_sample_image
img = load_sample_image("china.jpg").astype(float)
gray = img @ np.array([0.2989, 0.587, 0.114])    # RGB -> grayscale (a 2-D matrix)
print("image matrix shape:", gray.shape)

U, S, Vt = np.linalg.svd(gray, full_matrices=False)
def rank_k(k):
    return (U[:, :k] * S[:k]) @ Vt[:k, :]

ks = [5, 20, 50, len(S)]
fig, axes = plt.subplots(1, 4, figsize=(14, 4))
for ax, k in zip(axes, ks):
    ax.imshow(rank_k(k), cmap="gray")
    stored = k*(gray.shape[0] + gray.shape[1] + 1)
    ratio = gray.size / stored
    ax.set_title(f"k={k}  ({ratio:.1f}x smaller)" if k < len(S) else f"k={k} (full)")
    ax.axis("off")
plt.tight_layout(); plt.show()

In [ ]:
# How much 'energy' (variance) the top-k singular values capture
energy = np.cumsum(S**2) / np.sum(S**2)
fig, ax = plt.subplots(figsize=(6,3))
ax.plot(range(1, 61), energy[:60], marker=".")
ax.axhline(0.95, ls="--", color="gray")
ax.set_xlabel("number of singular values k"); ax.set_ylabel("fraction of energy")
ax.set_title("A few components capture most of the image"); ax.grid(alpha=0.3); plt.show()
print("k for 95% energy:", int(np.argmax(energy >= 0.95)) + 1)

**Exercise 5.** The `gray` image stands in for a microscopy field — a single-channel frame whose
structure sits in a handful of dominant modes, as a stained-tissue image does. Using the image and
its SVD from above, compute the reconstruction error (Frobenius norm of the difference) between the
rank-`k=20` approximation and the original, as a *fraction* of the image's own norm. Then find the
smallest `k` for which this relative error drops below 5%, and say what that `k` implies for
streaming frames off a microscope in real time.

> 🤖 *Gemini tip:* "Given the SVD of an image matrix in NumPy, show me how to compute the relative Frobenius-norm reconstruction error of a rank-k approximation and search for the smallest k under a tolerance."


In [ ]:
# Your code here

---
## 6 · Least squares as a projection, the pseudoinverse, and ridge (Géron Ch. 4)

Fitting `X w ≈ y` with more rows than unknowns is a **projection**: choose `w` so the residual is
orthogonal to every column of `X`, giving the **normal equation** `(X.T X) w = X.T y`. This is the
closed-form linear regression of Chapter 4. We solve it three ways and check they agree — then look at
what it is doing geometrically, at the pseudoinverse the libraries really use, and at what ridge
regularisation does to the spectrum.


In [ ]:
from sklearn.datasets import load_diabetes
from sklearn.linear_model import LinearRegression
d = load_diabetes()
X = np.c_[np.ones(len(d.data)), d.data]    # add a bias column
y = d.target

# (a) normal equation
w_normal = np.linalg.inv(X.T @ X) @ X.T @ y
# (b) np.linalg.lstsq (SVD-based, numerically safer)
w_lstsq, *_ = np.linalg.lstsq(X, y, rcond=None)
# (c) scikit-learn
lr = LinearRegression(fit_intercept=False).fit(X, y)

print("normal vs lstsq agree:", np.allclose(w_normal, w_lstsq))
print("lstsq vs sklearn agree:", np.allclose(w_lstsq, lr.coef_))
print("first 3 coefficients:", w_normal[:3])

**Exercise 6.** Compute the model's predictions `yhat = X @ w_normal` and report the R² score
`1 - SS_res/SS_tot`. Confirm it matches `lr.score(X, y)`.

> 🤖 *Gemini tip:* "Show me how to compute the R-squared score for a linear regression by hand from predictions and true values, and how it compares to scikit-learn's .score()."

In [ ]:
# Your code here

### 6.1 · Least squares *is* an orthogonal projection

Substituting the solution back into the prediction shows what the fit actually does to $\mathbf y$:

$$\hat{\mathbf y} \;=\; \mathbf X\mathbf w \;=\; \underbrace{\mathbf X(\mathbf X^{\mathsf T}\mathbf X)^{-1}\mathbf X^{\mathsf T}}_{\textstyle \mathbf P}\,\mathbf y,
\qquad
\mathbf r \;=\; \mathbf y - \hat{\mathbf y} \;=\; (\mathbf I - \mathbf P)\,\mathbf y .$$

$\mathbf P$ — the **hat matrix**, because it puts the hat on $\mathbf y$ — is the orthogonal projector
onto the **column space** of $\mathbf X$, the set of all vectors the model can possibly produce. Two
properties identify it: it is **symmetric** ($\mathbf P^{\mathsf T} = \mathbf P$) and **idempotent**
($\mathbf P^2 = \mathbf P$ — projecting something already in the plane changes nothing). Its rank is
the number of independent columns of $\mathbf X$. With a single column this collapses to the formula
of Notebook 1, $\mathbf P\mathbf y = \frac{\mathbf x^{\mathsf T}\mathbf y}{\mathbf x^{\mathsf T}\mathbf x}\mathbf x$:
least squares is that same projection, done in as many directions at once as you have features.


In [ ]:
# The hat matrix on the diabetes fit: symmetric, idempotent, and it kills the residual.
P = X @ np.linalg.inv(X.T @ X) @ X.T
yhat_P = P @ y
r = y - yhat_P

print("P is symmetric :", np.allclose(P, P.T))
print("P is idempotent:", np.allclose(P @ P, P))
print("rank(P)        :", np.linalg.matrix_rank(P), " = number of columns of X:", X.shape[1])
print("trace(P)       :", round(np.trace(P), 4), " (equals the rank, for a projector)")
print("P y equals X w :", np.allclose(yhat_P, X @ w_normal))
print("residual orthogonal to every column of X: max |X.T r| =", f"{np.abs(X.T @ r).max():.2e}")
print("Pythagoras: ||y||^2 =", round(y @ y, 2),
      " vs ||yhat||^2 + ||r||^2 =", round(yhat_P @ yhat_P + r @ r, 2))


In [ ]:
# The picture, in the smallest case that shows it: 3 patients, 2 columns -> a plane in R^3.
Xs3 = np.array([[1.0, 0.0],        # bias, and one centred predictor (heart rate)
                [1.0, 1.0],
                [1.0, 2.0]])
y3 = np.array([1.0, 3.0, 2.2])     # measured VO2, mL/kg/min
P3 = Xs3 @ np.linalg.inv(Xs3.T @ Xs3) @ Xs3.T
yh3, r3 = P3 @ y3, y3 - P3 @ y3
print("y      :", y3, "\nyhat   :", yh3.round(4), "\nresidual:", r3.round(4))
print("residual . column1 =", round(r3 @ Xs3[:, 0], 12),
      " residual . column2 =", round(r3 @ Xs3[:, 1], 12))

c1, c2 = Xs3[:, 0], Xs3[:, 1]
aa, bb = np.meshgrid(np.linspace(-0.6, 2.8, 16), np.linspace(-1.5, 1.5, 16))
plane = aa[..., None]*c1 + bb[..., None]*c2         # every vector the model can produce

fig = plt.figure(figsize=(7.0, 5.6))
ax = fig.add_subplot(111, projection="3d")
ax.plot_surface(plane[..., 0], plane[..., 1], plane[..., 2], alpha=0.22,
                color="#0E7C7B", linewidth=0)
ax.quiver(0, 0, 0, *y3,  color="crimson", lw=2.4, arrow_length_ratio=0.09)
ax.quiver(0, 0, 0, *yh3, color="#1B6CA8", lw=2.4, arrow_length_ratio=0.09)
ax.plot(*np.column_stack([yh3, y3]), color="#C77700", lw=2.4, ls="--")
ax.text(y3[0] - 0.05, y3[1] - 0.55, y3[2] + 0.12, "y  (measured)", color="crimson", fontsize=9)
ax.text(yh3[0], yh3[1] - 0.10, yh3[2] - 0.55, "yhat = P y", color="#1B6CA8", fontsize=9)
ax.text(*((yh3 + y3)/2 + np.array([0.0, 0.10, 0.28])), "residual", color="#C77700", fontsize=9)
ax.text(*(2.2*c1 - 1.2*c2 + np.array([0.0, 0.0, 0.12])), "column space of X",
        color="#0A5453", fontsize=8)
ax.set_xlim(-0.5, 3.0); ax.set_ylim(-0.5, 3.0); ax.set_zlim(-0.5, 3.0)
ax.set_xticks([0, 1, 2, 3]); ax.set_yticks([0, 1, 2, 3]); ax.set_zticks([0, 1, 2, 3])
ax.set_xlabel("patient 1", labelpad=-2); ax.set_ylabel("patient 2", labelpad=-2)
ax.set_zlabel("patient 3", labelpad=-2)
ax.tick_params(labelsize=8, pad=0)
ax.set_title("y, its shadow in the column space, and the perpendicular residual",
             fontsize=10, pad=-2)
ax.set_box_aspect((1, 1, 1), zoom=0.84); ax.view_init(elev=22, azim=-140)
plt.subplots_adjust(left=0.02, right=0.98, top=0.99, bottom=0.02); plt.show()


### 6.2 · The pseudoinverse: what `lstsq` actually calls

Forming $(\mathbf X^{\mathsf T}\mathbf X)^{-1}$ explicitly is the one thing you should not do in
production: squaring $\mathbf X$ **squares its condition number**, so a mildly correlated design
becomes a badly conditioned solve. The SVD gives the same answer without ever forming that product.
With $\mathbf X = \mathbf U\boldsymbol\Sigma\mathbf V^{\mathsf T}$, the **Moore–Penrose pseudoinverse**
is

$$\mathbf X^{+} \;=\; \mathbf V\,\boldsymbol\Sigma^{+}\,\mathbf U^{\mathsf T},
\qquad \boldsymbol\Sigma^{+} = \operatorname{diag}(1/\sigma_1,\dots,1/\sigma_r,0,\dots,0),$$

inverting the non-zero singular values and leaving the rest at zero. Then $\mathbf w = \mathbf X^{+}\mathbf y$
— which is exactly what `np.linalg.lstsq` and `sklearn`'s `LinearRegression` compute. When $\mathbf X$
is rank-deficient or has more unknowns than equations, the normal equation has no unique solution but
the pseudoinverse still returns one: the **minimum-norm** solution, the shortest $\mathbf w$ among all
that fit equally well.


In [ ]:
# Build the pseudoinverse by hand from the SVD and compare with the three other routes.
Uf, Sf, Vtf = np.linalg.svd(X, full_matrices=False)
Xplus = Vtf.T @ np.diag(1/Sf) @ Uf.T
w_pinv = Xplus @ y
print("hand-built X+ equals np.linalg.pinv:", np.allclose(Xplus, np.linalg.pinv(X)))
print("w from X+ equals w from lstsq      :", np.allclose(w_pinv, w_lstsq))

print("\ncondition number of X      :", f"{np.linalg.cond(X):.3e}")
print("condition number of X.T @ X:", f"{np.linalg.cond(X.T @ X):.3e}",
      " (squared -- this is why the normal equation is the fragile route)")

# an underdetermined problem: 3 equations, 5 unknowns -> infinitely many exact fits
rng = np.random.default_rng(0)
Xu = rng.normal(size=(3, 5)); yu = rng.normal(size=3)
w_min = np.linalg.pinv(Xu) @ yu
print("\nunderdetermined case: residual", f"{np.linalg.norm(Xu @ w_min - yu):.2e}",
      "| ||w|| =", round(np.linalg.norm(w_min), 4))
null_dirs = np.linalg.svd(Xu)[2][3:]                  # rows 3,4 of V.T span the null space of Xu
for k, ns in enumerate(null_dirs, 1):
    alt = w_min + 1.5*k*ns                            # add anything from the null space
    print(f"  alternative fit {k}: residual {np.linalg.norm(Xu @ alt - yu):.2e}"
          f" | ||w|| = {np.linalg.norm(alt):.4f}  (never shorter than the minimum-norm one)")


### 6.3 · Ridge, read through the eigenvalues

Ridge regression adds a penalty $\alpha\lVert\mathbf w\rVert_2^2$ to the squared error, which changes
the normal equation by one term:

$$\mathbf w_{\text{ridge}} \;=\; (\mathbf X^{\mathsf T}\mathbf X + \alpha\mathbf I)^{-1}\mathbf X^{\mathsf T}\mathbf y .$$

Since $\mathbf X^{\mathsf T}\mathbf X$ is symmetric positive semi-definite, it has an orthonormal
eigenbasis with eigenvalues $\lambda_i \ge 0$; adding $\alpha\mathbf I$ leaves the eigenvectors alone
and shifts every eigenvalue to $\lambda_i + \alpha$. So the condition number falls from
$\lambda_{\max}/\lambda_{\min}$ to $(\lambda_{\max}+\alpha)/(\lambda_{\min}+\alpha)$, a singular
problem becomes solvable, and the directions that are shrunk hardest are exactly the low-variance ones
— the ones the data says least about. That is regularisation in one line of linear algebra.


In [ ]:
# What alpha does to the spectrum, the conditioning, and the coefficients.
G = X.T @ X
lam = np.linalg.eigvalsh(G)
print("eigenvalues of X.T X: min", f"{lam.min():.3e}", " max", f"{lam.max():.3e}",
      " condition number", f"{lam.max()/lam.min():.1f}")
for alpha in [0.0, 0.01, 0.1, 1.0]:
    cond = (lam.max() + alpha)/(lam.min() + alpha)
    w_a = np.linalg.solve(G + alpha*np.eye(G.shape[0]), X.T @ y)
    print(f"  alpha = {alpha:5.2f}: condition number {cond:8.1f} | ||w|| = {np.linalg.norm(w_a):8.2f}"
          f" | training RMSE = {np.sqrt(np.mean((X @ w_a - y)**2)):.3f}")

alphas = np.logspace(-4, 3, 60)
paths = np.array([np.linalg.solve(G + a*np.eye(G.shape[0]), X.T @ y) for a in alphas])
fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 3.8))
for j in range(1, X.shape[1]):                      # skip the bias column
    a1.semilogx(alphas, paths[:, j], lw=1.4)
a1.set_xlabel("alpha"); a1.set_ylabel("coefficient")
a1.set_title("Ridge path: every coefficient shrinks"); a1.grid(alpha=0.3)
a2.loglog(alphas, (lam.max() + alphas)/(lam.min() + alphas), color="#0E7C7B")
a2.set_xlabel("alpha"); a2.set_ylabel("condition number of X.T X + alpha I")
a2.set_title("and the problem becomes better conditioned"); a2.grid(alpha=0.3, which="both")
plt.tight_layout(); plt.show()


**Exercise 7.** Stay with the diabetes design matrix `X` and target `y`. (a) Build the hat matrix
`P` and verify numerically that `P @ P` equals `P` and that `trace(P)` equals `rank(P)`. (b) Compute
`w` three ways — normal equation, `np.linalg.pinv(X) @ y`, and `np.linalg.lstsq` — and confirm all
three agree. (c) Compute the ridge solution for `alpha = 1.0` and report by what factor the norm of
the coefficient vector has shrunk relative to `alpha = 0`. Which is larger, the drop in `||w||` or the
rise in training RMSE?

> 🤖 *Gemini tip:* "Explain why the least-squares hat matrix X(X^T X)^-1 X^T is an orthogonal projector, what its trace tells you, and show me NumPy code comparing the normal equation, the pseudoinverse and lstsq on the same design matrix."


In [ ]:
alpha_q = 1.0

# Your code here

---
## 7 · PCA on the breast-cancer data (Géron Ch. 7)

**PCA** rotates the feature axes to an orthonormal basis ordered by variance — a change of basis found
by the SVD, in exactly the $\boldsymbol\Gamma_{BE}$ sense of Notebook 1. Three things have to be right:

**Centre first.** PCA explains *variance*, which is defined about the mean. Skip the centring and the
first component chases the mean vector instead of the spread — the single most common PCA bug.
Standardising (centre *and* divide by the standard deviation) additionally stops whichever feature
happens to be measured in large units from dominating.

**The object being diagonalised is the covariance matrix.** For a centred $m\times n$ matrix
$\mathbf X$, $\mathbf C = \frac1m\mathbf X^{\mathsf T}\mathbf X$ is symmetric and positive
semi-definite (since $\mathbf v^{\mathsf T}\mathbf C\mathbf v = \frac1m\lVert\mathbf X\mathbf v\rVert^2 \ge 0$),
so the spectral theorem of Section 3 guarantees an orthonormal eigenbasis with non-negative
eigenvalues. Those eigenvectors are the principal components.

**The SVD gives them without forming $\mathbf C$.** If $\mathbf X = \mathbf U\boldsymbol\Sigma\mathbf V^{\mathsf T}$
then $\mathbf C = \frac1m\mathbf V\boldsymbol\Sigma^2\mathbf V^{\mathsf T}$, so the components are the
columns of $\mathbf V$ and $\lambda_i = \sigma_i^2/m$. The **explained variance ratio** of component
$i$ is $\sigma_i^2/\sum_j\sigma_j^2$, projection onto the top $k$ is $\mathbf X\mathbf V_k$, and
reconstruction is $\mathbf X\mathbf V_k\mathbf V_k^{\mathsf T}$.


In [ ]:
from sklearn.datasets import load_breast_cancer
bc = load_breast_cancer()
Xs = (bc.data - bc.data.mean(0)) / bc.data.std(0)     # standardise (Module 1)

U, S, Vt = np.linalg.svd(Xs, full_matrices=False)
scores = U[:, :2] * S[:2]                              # 2-D PCA projection
explained = (S**2 / np.sum(S**2))
print("variance explained by PC1, PC2:", explained[:2].round(3),
      "| cumulative:", explained[:2].sum().round(3))

fig, ax = plt.subplots(figsize=(6,5))
for cls, name, col in [(0,"malignant","crimson"), (1,"benign","steelblue")]:
    m = bc.target == cls
    ax.scatter(scores[m,0], scores[m,1], s=12, alpha=0.6, color=col, label=name)
ax.set_xlabel("PC1"); ax.set_ylabel("PC2"); ax.legend()
ax.set_title("Breast-cancer data in its first two principal components"); plt.show()

In [ ]:
# Confirm against scikit-learn (signs of components are arbitrary, so compare magnitudes)
from sklearn.decomposition import PCA
pca = PCA(n_components=2).fit(Xs)
print("sklearn explained variance ratio:", pca.explained_variance_ratio_.round(3))
print("matches our SVD values:", np.allclose(pca.explained_variance_ratio_, explained[:2]))

In [ ]:
# Centring is not optional -- and it is easiest to see in two dimensions.
rngc = np.random.default_rng(3)
t = rngc.normal(size=300)
cloud = np.column_stack([130 + 6.0*t + 2.0*rngc.normal(size=300),    # systolic BP, mmHg
                         70  - 6.0*t + 2.0*rngc.normal(size=300)])   # heart rate, bpm

pc1_raw = np.linalg.svd(cloud, full_matrices=False)[2][0]            # no centring
pc1_ctr = np.linalg.svd(cloud - cloud.mean(0), full_matrices=False)[2][0]
mu = cloud.mean(0); mu_dir = mu/np.linalg.norm(mu)
pc1_raw = np.sign(pc1_raw @ mu_dir)*pc1_raw                          # fix the arbitrary sign
print("mean of the cloud      :", mu.round(2))
print("uncentred PC1          :", pc1_raw.round(3),
      " | alignment with the mean direction:", round(abs(pc1_raw @ mu_dir), 4))
print("centred   PC1          :", pc1_ctr.round(3),
      " | alignment with the mean direction:", round(abs(pc1_ctr @ mu_dir), 4))
print("-> without centring, PC1 points at WHERE the cloud sits; with it, at HOW the cloud is shaped.")

fig, (g1, g2) = plt.subplots(1, 2, figsize=(11, 4.4))
g1.scatter(cloud[:, 0], cloud[:, 1], s=8, alpha=0.45, color="#0E7C7B")
g1.scatter(*mu, color="black", s=40, zorder=5)
g1.annotate("", xy=165*pc1_raw, xytext=(0, 0),
            arrowprops=dict(arrowstyle="-|>", color="#B23A48", lw=2))
g1.set_xlim(0, 175); g1.set_ylim(0, 110); g1.set_aspect("equal")
g1.set_title("uncentred: PC1 runs from the ORIGIN to the cloud", fontsize=10)

g2.scatter(cloud[:, 0], cloud[:, 1], s=10, alpha=0.5, color="#0E7C7B")
g2.scatter(*mu, color="black", s=45, zorder=5, label="mean")
g2.annotate("", xy=mu + 22*pc1_ctr, xytext=mu - 22*pc1_ctr,
            arrowprops=dict(arrowstyle="<|-|>", color="#1B6CA8", lw=2.2))
g2.annotate("", xy=mu + 22*pc1_raw, xytext=mu - 22*pc1_raw,
            arrowprops=dict(arrowstyle="<|-|>", color="#B23A48", lw=1.6, ls="dashed"))
g2.plot([], [], color="#1B6CA8", lw=2, label="centred PC1 (the real spread)")
g2.plot([], [], color="#B23A48", lw=1.6, ls="--", label="uncentred PC1, for comparison")
g2.set_aspect("equal"); g2.legend(fontsize=8, loc="upper right")
g2.set_title("zoomed on the cloud", fontsize=10)
for g in (g1, g2):
    g.set_xlabel("systolic BP (mmHg)"); g.set_ylabel("heart rate (bpm)")
plt.tight_layout(); plt.show()


In [ ]:
# And the same effect on the real 30-feature data, plus what standardising adds on top.
raw = bc.data
ev = lambda M: (lambda sv: sv**2/np.sum(sv**2))(np.linalg.svd(M, full_matrices=False)[1])
print("explained by PC1, raw (not centred) :", round(ev(raw)[0], 4))
print("explained by PC1, centred only      :", round(ev(raw - raw.mean(0))[0], 4))
print("explained by PC1, standardised (Xs) :", round(ev(Xs)[0], 4))
print("\nCentring stops PC1 chasing the mean; standardising stops whichever feature happens to be")
print("measured in the largest units (here 'area', in the hundreds) from owning the first component.")


In [ ]:
# Covariance eigenvalues and SVD singular values are the same information: lambda_i = sigma_i^2 / m.
m = Xs.shape[0]
C = (Xs.T @ Xs)/m                                        # Xs is the standardised matrix from above
lam_C, V_C = np.linalg.eigh(C)                           # eigh: ascending order, orthonormal columns
lam_C, V_C = lam_C[::-1], V_C[:, ::-1]                   # put the largest first

print("C symmetric      :", np.allclose(C, C.T))
print("C PSD (all lambda >= 0):", bool((lam_C > -1e-12).all()))
print("lambda_i  (top 4):", lam_C[:4].round(5))
print("sigma_i^2/m (top 4):", (S[:4]**2/m).round(5))
print("they match:", np.allclose(lam_C[:len(S)], S**2/m))
print("components match up to sign:",
      np.allclose(np.abs(V_C[:, :3]), np.abs(Vt[:3].T)))
print("\nexplained variance ratio, top 5:", explained[:5].round(4))
print("cumulative                   :", np.cumsum(explained)[:5].round(4))


In [ ]:
# Projection and reconstruction: X Vk (scores) and X Vk Vk.T (the best rank-k picture of the data).
def reconstruct(k):
    Vk = Vt[:k].T                                        # n x k, orthonormal columns
    return Xs @ Vk @ Vk.T

ks = [1, 2, 5, 10, 20, 30]
tot = np.sum(Xs**2)
print(f"{'k':>3} {'explained':>10} {'rel. recon. error':>19}")
for k in ks:
    err = np.linalg.norm(Xs - reconstruct(k))/np.sqrt(tot)
    print(f"{k:>3} {np.cumsum(explained)[k-1]:>10.4f} {err:>19.4f}")

kk = np.arange(1, len(explained) + 1)
fig, (b1, b2) = plt.subplots(1, 2, figsize=(11, 3.6))
b1.bar(kk, explained, color="#0E7C7B")
b1.plot(kk, np.cumsum(explained), "o-", ms=3, color="#C77700", label="cumulative")
b1.axhline(0.90, color="crimson", ls="--", lw=1, label="90%")
b1.set_xlabel("component"); b1.set_ylabel("explained variance ratio")
b1.set_title("Scree plot"); b1.legend(fontsize=8)
b2.plot(kk, [np.linalg.norm(Xs - reconstruct(k))/np.sqrt(tot) for k in kk], "o-",
        ms=3, color="#1B6CA8")
b2.set_xlabel("components kept k"); b2.set_ylabel("relative reconstruction error")
b2.set_title("What you give up by keeping only k"); b2.grid(alpha=0.3)
plt.tight_layout(); plt.show()


**Exercise 8.** How many principal components are needed to capture at least 90% of the variance in
the standardised breast-cancer features? *Hint:* cumulative sum of `explained`.

> 🤖 *Gemini tip:* "Given an array of per-component explained-variance ratios from PCA, show me how to find the smallest number of components whose cumulative variance reaches a target threshold."

In [ ]:
# Your code here

**Exercise 9.** A hospital wants to store the 30 standardised breast-cancer features per patient in a
compressed form. (a) Find the smallest `k` whose reconstruction `Xs @ Vk @ Vk.T` keeps the relative
error below 0.25, and report the compression ratio (numbers stored per patient, `k` instead of 30).
(b) Confirm that the reconstruction error and the explained variance are two views of one number by
checking that the relative error squared equals `1 - cumulative_explained[k-1]`. (c) Reconstruct the
first patient's record with that `k` and print the three features whose values moved most.

> 🤖 *Gemini tip:* "Given the SVD of a standardised data matrix, show me how to reconstruct it from the top k principal components, how the relative reconstruction error relates to the cumulative explained variance ratio, and how to compare an original row with its reconstruction."


In [ ]:
target_err = 0.25

# Your code here

---
### Module 2 complete
Eigenvectors and diagonalisation, the spectral theorem, the SVD and low-rank approximation, least
squares as an orthogonal projection onto the column space — with its hat matrix, its pseudoinverse and
ridge read through the eigenvalues — and PCA as an SVD change of basis, from centring through
explained variance to reconstruction. That is the linear algebra Géron's Chapters 4 and 7 run on, plus
the $SO(3)$/rotation algebra shared with the Robotics unit. **Next:** Module 3 (Calculus) turns to
derivatives, gradients and the integral behind the PID controller.
